In [0]:
WITH
silver_trip_stats AS (
  SELECT COUNT(*) AS row_count
  FROM nyc_taxi.silver.silver_yellow_trip_2025
),

silver_zone_stats AS (
  SELECT COUNT(*) AS row_count
  FROM nyc_taxi.silver.silver_taxi_zone_lookup
),

fact_stats AS (
  SELECT
    COUNT(*) AS row_count,
    COUNT_IF(trip_key IS NULL OR TRIM(trip_key) = '') AS null_trip_keys,
    COUNT_IF(
      pickup_date_key IS NULL
      OR dropoff_date_key IS NULL
      OR pickup_time_key IS NULL
      OR dropoff_time_key IS NULL
      OR pickup_zone_key IS NULL
      OR dropoff_zone_key IS NULL
      OR payment_type_key IS NULL
      OR rate_code_key IS NULL
      OR vendor_key IS NULL
    ) AS null_dimension_keys,
    COUNT_IF(trip_count <> 1 OR trip_count IS NULL) AS invalid_trip_count,
    COUNT_IF(trip_duration_minutes <= 0 OR trip_duration_minutes IS NULL)
      AS invalid_duration_rows,
    COUNT_IF(
      pickup_date_key = 0
      OR dropoff_date_key = 0
      OR pickup_time_key = -1
      OR dropoff_time_key = -1
      OR pickup_zone_key = 0
      OR dropoff_zone_key = 0
    ) AS unknown_critical_dimension_rows,
    COUNT_IF(
      payment_type_key = -1
      OR rate_code_key = -1
      OR vendor_key = -1
    ) AS unknown_business_code_rows,
    COUNT_IF(NOT is_efficiency_kpi_eligible) AS efficiency_excluded_rows,
    COUNT_IF(
      total_amount < 0
      OR fare_amount < 0
      OR tip_amount < 0
      OR tolls_amount < 0
    ) AS negative_financial_rows,
    COUNT_IF(
      ABS(
        total_amount - (
          fare_amount
          + extra_amount
          + mta_tax_amount
          + tip_amount
          + tolls_amount
          + improvement_surcharge_amount
          + congestion_surcharge_amount
          + airport_fee_amount
          + cbd_congestion_fee_amount
        )
      ) > 0.01
    ) AS financial_reconciliation_rows,
    COUNT_IF(
      route_key <> CONCAT(
        CAST(pickup_zone_key AS STRING),
        ':',
        CAST(dropoff_zone_key AS STRING)
      )
      OR route_key IS NULL
    ) AS invalid_route_keys
  FROM nyc_taxi.gold.fact_yellow_taxi_trip
),

duplicate_trip_stats AS (
  SELECT
    COUNT(*) AS duplicate_key_count,
    COALESCE(SUM(key_occurrences - 1), 0) AS duplicate_extra_rows
  FROM (
    SELECT
      trip_key,
      COUNT(*) AS key_occurrences
    FROM nyc_taxi.gold.fact_yellow_taxi_trip
    GROUP BY trip_key
    HAVING COUNT(*) > 1
  ) duplicates
),

date_stats AS (
  SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT date_key) AS distinct_key_count,
    COUNT_IF(date_key = 0) AS unknown_member_count,
    COUNT_IF(date_key <> 0 AND full_date IS NULL) AS invalid_member_count
  FROM nyc_taxi.gold.dim_date
),

time_stats AS (
  SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT time_key) AS distinct_key_count,
    COUNT_IF(time_key = -1) AS unknown_member_count,
    COUNT_IF(time_key <> -1 AND (hour_number < 0 OR hour_number > 23))
      AS invalid_member_count
  FROM nyc_taxi.gold.dim_time
),

pickup_zone_stats AS (
  SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT pickup_zone_key) AS distinct_key_count,
    COUNT_IF(pickup_zone_key = 0) AS unknown_member_count
  FROM nyc_taxi.gold.dim_pickup_zone
),

dropoff_zone_stats AS (
  SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT dropoff_zone_key) AS distinct_key_count,
    COUNT_IF(dropoff_zone_key = 0) AS unknown_member_count
  FROM nyc_taxi.gold.dim_dropoff_zone
),

payment_stats AS (
  SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT payment_type_key) AS distinct_key_count,
    COUNT_IF(payment_type_key = -1) AS unknown_member_count
  FROM nyc_taxi.gold.dim_payment_type
),

rate_stats AS (
  SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT rate_code_key) AS distinct_key_count,
    COUNT_IF(rate_code_key = -1) AS unknown_member_count
  FROM nyc_taxi.gold.dim_rate_code
),

vendor_stats AS (
  SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT vendor_key) AS distinct_key_count,
    COUNT_IF(vendor_key = -1) AS unknown_member_count
  FROM nyc_taxi.gold.dim_vendor
),

foreign_key_stats AS (
  SELECT
    COUNT_IF(pickup_date.date_key IS NULL) AS orphan_pickup_date,
    COUNT_IF(dropoff_date.date_key IS NULL) AS orphan_dropoff_date,
    COUNT_IF(pickup_time.time_key IS NULL) AS orphan_pickup_time,
    COUNT_IF(dropoff_time.time_key IS NULL) AS orphan_dropoff_time,
    COUNT_IF(pickup_zone.pickup_zone_key IS NULL) AS orphan_pickup_zone,
    COUNT_IF(dropoff_zone.dropoff_zone_key IS NULL) AS orphan_dropoff_zone,
    COUNT_IF(payment.payment_type_key IS NULL) AS orphan_payment,
    COUNT_IF(rate.rate_code_key IS NULL) AS orphan_rate,
    COUNT_IF(vendor.vendor_key IS NULL) AS orphan_vendor
  FROM nyc_taxi.gold.fact_yellow_taxi_trip fact
  LEFT JOIN nyc_taxi.gold.dim_date pickup_date
    ON fact.pickup_date_key = pickup_date.date_key
  LEFT JOIN nyc_taxi.gold.dim_date dropoff_date
    ON fact.dropoff_date_key = dropoff_date.date_key
  LEFT JOIN nyc_taxi.gold.dim_time pickup_time
    ON fact.pickup_time_key = pickup_time.time_key
  LEFT JOIN nyc_taxi.gold.dim_time dropoff_time
    ON fact.dropoff_time_key = dropoff_time.time_key
  LEFT JOIN nyc_taxi.gold.dim_pickup_zone pickup_zone
    ON fact.pickup_zone_key = pickup_zone.pickup_zone_key
  LEFT JOIN nyc_taxi.gold.dim_dropoff_zone dropoff_zone
    ON fact.dropoff_zone_key = dropoff_zone.dropoff_zone_key
  LEFT JOIN nyc_taxi.gold.dim_payment_type payment
    ON fact.payment_type_key = payment.payment_type_key
  LEFT JOIN nyc_taxi.gold.dim_rate_code rate
    ON fact.rate_code_key = rate.rate_code_key
  LEFT JOIN nyc_taxi.gold.dim_vendor vendor
    ON fact.vendor_key = vendor.vendor_key
),

checks AS (
  SELECT
    10 AS check_order,
    'fact_yellow_taxi_trip' AS asset_name,
    'Quantidade da fato igual à Silver' AS check_name,
    CASE WHEN fact.row_count = silver.row_count THEN 'PASS' ELSE 'FAIL' END AS status,
    CAST(fact.row_count AS STRING) AS actual_value,
    CAST(silver.row_count AS STRING) AS expected_value,
    'A Gold deve manter uma linha para cada viagem válida da Silver.' AS details
  FROM fact_stats fact
  CROSS JOIN silver_trip_stats silver

  UNION ALL

  SELECT
    20,
    'fact_yellow_taxi_trip',
    'Trip key preenchida',
    CASE WHEN null_trip_keys = 0 THEN 'PASS' ELSE 'FAIL' END,
    CAST(null_trip_keys AS STRING),
    '0',
    'Não pode haver trip_key nula ou vazia.'
  FROM fact_stats

  UNION ALL

  SELECT
    30,
    'fact_yellow_taxi_trip',
    'Unicidade da trip key',
    CASE WHEN duplicate_key_count = 0 THEN 'PASS' ELSE 'FAIL' END,
    CAST(duplicate_key_count AS STRING),
    '0',
    CONCAT('Linhas excedentes associadas a chaves duplicadas: ', duplicate_extra_rows)
  FROM duplicate_trip_stats

  UNION ALL

  SELECT
    40,
    'fact_yellow_taxi_trip',
    'Chaves dimensionais preenchidas',
    CASE WHEN null_dimension_keys = 0 THEN 'PASS' ELSE 'FAIL' END,
    CAST(null_dimension_keys AS STRING),
    '0',
    'Valores ausentes devem utilizar o membro Unknown, e não NULL.'
  FROM fact_stats

  UNION ALL

  SELECT
    50,
    'fact_yellow_taxi_trip',
    'Integridade referencial',
    CASE
      WHEN (
        orphan_pickup_date + orphan_dropoff_date
        + orphan_pickup_time + orphan_dropoff_time
        + orphan_pickup_zone + orphan_dropoff_zone
        + orphan_payment + orphan_rate + orphan_vendor
      ) = 0 THEN 'PASS'
      ELSE 'FAIL'
    END,
    CAST(
      orphan_pickup_date + orphan_dropoff_date
      + orphan_pickup_time + orphan_dropoff_time
      + orphan_pickup_zone + orphan_dropoff_zone
      + orphan_payment + orphan_rate + orphan_vendor
      AS STRING
    ),
    '0',
    CONCAT(
      'pickup_date=', orphan_pickup_date,
      '; dropoff_date=', orphan_dropoff_date,
      '; pickup_time=', orphan_pickup_time,
      '; dropoff_time=', orphan_dropoff_time,
      '; pickup_zone=', orphan_pickup_zone,
      '; dropoff_zone=', orphan_dropoff_zone,
      '; payment=', orphan_payment,
      '; rate=', orphan_rate,
      '; vendor=', orphan_vendor
    )
  FROM foreign_key_stats

  UNION ALL

  SELECT
    60,
    'fact_yellow_taxi_trip',
    'Trip count aditivo',
    CASE WHEN invalid_trip_count = 0 THEN 'PASS' ELSE 'FAIL' END,
    CAST(invalid_trip_count AS STRING),
    '0',
    'Cada linha da fato deve possuir trip_count = 1.'
  FROM fact_stats

  UNION ALL

  SELECT
    70,
    'fact_yellow_taxi_trip',
    'Duração positiva',
    CASE WHEN invalid_duration_rows = 0 THEN 'PASS' ELSE 'FAIL' END,
    CAST(invalid_duration_rows AS STRING),
    '0',
    'A Silver validada não deve produzir duração nula ou não positiva.'
  FROM fact_stats

  UNION ALL

  SELECT
    80,
    'fact_yellow_taxi_trip',
    'Consistência da route key',
    CASE WHEN invalid_route_keys = 0 THEN 'PASS' ELSE 'FAIL' END,
    CAST(invalid_route_keys AS STRING),
    '0',
    'route_key deve ser pickup_zone_key:dropoff_zone_key.'
  FROM fact_stats

  UNION ALL

  SELECT
    90,
    'fact_yellow_taxi_trip',
    'Uso de Unknown em dimensões críticas',
    CASE WHEN unknown_critical_dimension_rows = 0 THEN 'PASS' ELSE 'WARN' END,
    CAST(unknown_critical_dimension_rows AS STRING),
    '0',
    'Investigar datas, horários ou zonas que não encontraram dimensão.'
  FROM fact_stats

  UNION ALL

  SELECT
    100,
    'fact_yellow_taxi_trip',
    'Códigos de negócio não mapeados',
    CASE WHEN unknown_business_code_rows = 0 THEN 'PASS' ELSE 'WARN' END,
    CAST(unknown_business_code_rows AS STRING),
    '0',
    'Pode refletir valores nulos ou códigos novos de pagamento, tarifa ou vendor.'
  FROM fact_stats

  UNION ALL

  SELECT
    110,
    'fact_yellow_taxi_trip',
    'Viagens excluídas dos KPIs de eficiência',
    CASE WHEN efficiency_excluded_rows = 0 THEN 'PASS' ELSE 'WARN' END,
    CAST(efficiency_excluded_rows AS STRING),
    '0',
    'Viagens sem distância e duração positivas não devem compor velocidade média.'
  FROM fact_stats

  UNION ALL

  SELECT
    120,
    'fact_yellow_taxi_trip',
    'Valores financeiros negativos',
    CASE WHEN negative_financial_rows = 0 THEN 'PASS' ELSE 'WARN' END,
    CAST(negative_financial_rows AS STRING),
    '0',
    'Valores negativos podem representar ajustes; confirmar a regra do negócio.'
  FROM fact_stats

  UNION ALL

  SELECT
    130,
    'fact_yellow_taxi_trip',
    'Reconciliação dos componentes financeiros',
    CASE WHEN financial_reconciliation_rows = 0 THEN 'PASS' ELSE 'WARN' END,
    CAST(financial_reconciliation_rows AS STRING),
    '0',
    'Compara total_amount com a soma dos componentes usando tolerância de US$ 0,01.'
  FROM fact_stats

  UNION ALL

  SELECT
    200,
    'dim_date',
    'Calendário completo e chave única',
    CASE
      WHEN row_count = 366
        AND distinct_key_count = 366
        AND unknown_member_count = 1
        AND invalid_member_count = 0
      THEN 'PASS' ELSE 'FAIL'
    END,
    CAST(row_count AS STRING),
    '366',
    '365 datas de 2025 mais o membro Unknown de chave 0.'
  FROM date_stats

  UNION ALL

  SELECT
    210,
    'dim_time',
    'Horas completas e chave única',
    CASE
      WHEN row_count = 25
        AND distinct_key_count = 25
        AND unknown_member_count = 1
        AND invalid_member_count = 0
      THEN 'PASS' ELSE 'FAIL'
    END,
    CAST(row_count AS STRING),
    '25',
    '24 horas mais o membro Unknown de chave -1.'
  FROM time_stats

  UNION ALL

  SELECT
    220,
    'dim_pickup_zone',
    'Cobertura e unicidade das zonas de embarque',
    CASE
      WHEN pickup.row_count = silver.row_count + 1
        AND pickup.row_count = pickup.distinct_key_count
        AND pickup.unknown_member_count = 1
      THEN 'PASS' ELSE 'FAIL'
    END,
    CAST(pickup.row_count AS STRING),
    CAST(silver.row_count + 1 AS STRING),
    'Todas as zonas Silver mais o membro Unknown de chave 0.'
  FROM pickup_zone_stats pickup
  CROSS JOIN silver_zone_stats silver

  UNION ALL

  SELECT
    230,
    'dim_dropoff_zone',
    'Cobertura e unicidade das zonas de desembarque',
    CASE
      WHEN dropoff.row_count = silver.row_count + 1
        AND dropoff.row_count = dropoff.distinct_key_count
        AND dropoff.unknown_member_count = 1
      THEN 'PASS' ELSE 'FAIL'
    END,
    CAST(dropoff.row_count AS STRING),
    CAST(silver.row_count + 1 AS STRING),
    'Todas as zonas Silver mais o membro Unknown de chave 0.'
  FROM dropoff_zone_stats dropoff
  CROSS JOIN silver_zone_stats silver

  UNION ALL

  SELECT
    240,
    'dim_payment_type',
    'Domínio de pagamentos e chave única',
    CASE
      WHEN row_count = 8
        AND distinct_key_count = 8
        AND unknown_member_count = 1
      THEN 'PASS' ELSE 'FAIL'
    END,
    CAST(row_count AS STRING),
    '8',
    'Unknown técnico, Flex Fare e códigos TLC de 1 a 6.'
  FROM payment_stats

  UNION ALL

  SELECT
    250,
    'dim_rate_code',
    'Domínio de tarifas e chave única',
    CASE
      WHEN row_count = 8
        AND distinct_key_count = 8
        AND unknown_member_count = 1
      THEN 'PASS' ELSE 'FAIL'
    END,
    CAST(row_count AS STRING),
    '8',
    'Unknown técnico, códigos TLC de 1 a 6 e código 99.'
  FROM rate_stats

  UNION ALL

  SELECT
    260,
    'dim_vendor',
    'Domínio de vendors e chave única',
    CASE
      WHEN row_count = 5
        AND distinct_key_count = 5
        AND unknown_member_count = 1
      THEN 'PASS' ELSE 'FAIL'
    END,
    CAST(row_count AS STRING),
    '5',
    'Unknown técnico e vendors TLC 1, 2, 6 e 7.'
  FROM vendor_stats
)

SELECT
  asset_name,
  check_name,
  status,
  actual_value,
  expected_value,
  details
FROM checks
ORDER BY check_order;
